In [ ]:
import torch
print("GPU Aktif mi?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Ekran Kartı:", torch.cuda.get_device_name(0))

GPU Aktif mi?: True
Ekran Kartı: Tesla T4


In [ ]:
%pip install ultralytics tensorrt lap

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 77.1 MB/s eta 0:00:00
  Created wheel for tensorrt: filename=tensorrt-11.2.1.2-py3-none-any.whl size=16537 sha256=af29eb12b06dd69fd5eb31be9b5595ac3a9d2d2bf156c6d98f766598f5532595
  Stored in directory: /root/.cache/pip/wheels/74/d7/1e/ef122bffd3247ebd1bd40aa1b521e2aa7aa6bf6d4c38931254
  Created wheel for 

In [ ]:
import cv2
import time
import os
import csv
import numpy as np
from datetime import datetime
from ultralytics import YOLO

# 1. ADIM 4.3 PIPELINE (TRACKING + ANOMALY LOGGING)
class EdgeVideoPipelineWithTracking:
    def __init__(self, engine_path, min_aspect_ratio=0.4, max_human_pixel=254, temp_threshold=40.0, output_dir="/content/anomalies"):
        # INT8 Engine Yükleniyor
        self.model = YOLO(engine_path, task="detect")
        self.min_aspect_ratio = min_aspect_ratio
        self.max_human_pixel = max_human_pixel
        self.temp_threshold = temp_threshold
        self.output_dir = output_dir
        self.clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))

        # Anomali Görsel ve Log Klasörünün Hazırlanması
        os.makedirs(self.output_dir, exist_ok=True)
        self.csv_path = os.path.join(self.output_dir, "anomali_log.csv")

        # Aynı kişi için sürekli log/resim üretmemek için kayıtlı ID'ler kümesi
        self.logged_ids = set()

        # CSV Log Dosyası Başlıklarını Oluştur
        if not os.path.exists(self.csv_path):
            with open(self.csv_path, mode="w", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerow(["Timestamp", "Track_ID", "Temperature_C", "BBox_x1_y1_x2_y2"])

    def process_frame(self, gray_frame):
        enhanced_gray = self.clahe.apply(gray_frame)
        inferno_frame = cv2.applyColorMap(enhanced_gray, cv2.COLORMAP_INFERNO)

        # ByteTrack ile Nesne Takibi (persist=True kareler arası kimliği korur)
        results = self.model.track(
            enhanced_gray,
            persist=True,
            tracker="bytetrack.yaml",
            conf=0.30,
            iou=0.50,
            verbose=False
        )[0]

        if results.boxes is not None:
            for box in results.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                w, h = x2 - x1, y2 - y1

                # Geometri ve Sıcaklık Tavan Filtresi
                if w <= 0 or h <= 0 or (h / float(w)) < self.min_aspect_ratio:
                    continue

                roi = enhanced_gray[y1:y2, x1:x2]
                if roi.size == 0 or np.max(roi) > self.max_human_pixel:
                    continue

                # Nesne Takip ID'si (Atanmadıysa N/A)
                track_id = int(box.id[0].item()) if box.id is not None else "N/A"

                # Sıcaklık Hesabı (%90 Percentile)
                top_10 = np.percentile(roi, 90)
                avg_heat = np.mean(roi[roi >= top_10])
                temp = round(float(30.0 + (avg_heat * 0.03)), 1)
                is_anomaly = temp >= self.temp_threshold

                # --- ANOMALİ LOGLAMA VE SAKLAMA MEKANİZMASI ---
                if is_anomaly and track_id != "N/A":
                    # Bu ID daha önce loglanmadıysa 1 defa kaydet
                    if track_id not in self.logged_ids:
                        self.logged_ids.add(track_id)
                        timestamp_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

                        # 1. CSV Log Kaydı
                        with open(self.csv_path, mode="a", newline="", encoding="utf-8") as f:
                            writer = csv.writer(f)
                            writer.writerow([timestamp_str, f"ID_{track_id}", temp, [x1, y1, x2, y2]])

                        # 2. Anomali Anı Ekran Görüntüsü Kaydı
                        snapshot_path = os.path.join(self.output_dir, f"anomali_ID{track_id}_{temp}C.jpg")
                        cv2.imwrite(snapshot_path, inferno_frame)
                        print(f"[ALARM] Anomali Tespit Edildi! ID: {track_id} | Sıcaklık: {temp}°C -> {snapshot_path}")

                # Çizim (Normal: Yeşil, Anomali: Kırmızı)
                color = (0, 0, 255) if is_anomaly else (0, 255, 0)
                label = f"ID:{track_id} | {temp}C"
                cv2.rectangle(inferno_frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(inferno_frame, label, (x1, max(y1-5, 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        return inferno_frame

# 2. VİDEO ÇALIŞTIRMA VE TEST
VIDEO_PATH = "/content/test_thermal_video.mp4"
OUTPUT_VIDEO_PATH = "/content/processed_thermal_video_tracked.mp4"
ENGINE_PATH = "/content/yolov8_gold_best_int8.engine"

# NOT: Test aşamasında anomali sisteminin ve logların çalıştığını görmek için
# temp_threshold değerini geçici olarak 36.5 yapabilirsiniz.
pipeline = EdgeVideoPipelineWithTracking(
    engine_path=ENGINE_PATH,
    temp_threshold=36.5,  # Test için düşürüldü (Normalde 40.0)
    output_dir="/content/anomalies"
)

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    print(f"Video açılamadı: {VIDEO_PATH}")
else:
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    PLAYBACK_FPS = 10.0

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, PLAYBACK_FPS, (width, height))

    frame_count = 0
    start_time = time.time()

    print("Edge Video Pipeline (Tracking & Logging) çalışıyor...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) if len(frame.shape) == 3 else frame
        processed_frame = pipeline.process_frame(gray_frame)

        frame_count += 1
        elapsed_time = time.time() - start_time
        current_fps = frame_count / elapsed_time if elapsed_time > 0 else 0

        cv2.putText(processed_frame, f"Edge Processing FPS: {current_fps:.1f}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        out.write(processed_frame)

    cap.release()
    out.release()
    print(f"\n Video işlendi: {OUTPUT_VIDEO_PATH}")
    print(f"Ortalama Canlı İşleme Hızı: {current_fps:.1f} FPS")

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Edge Video Pipeline (Tracking & Logging) çalışıyor...
Loading /content/yolov8_gold_best_int8.engine for TensorRT inference...
[ALARM] Anomali Tespit Edildi! ID: 6 | Sıcaklık: 36.8°C -> /content/anomalies/anomali_ID6_36.8C.jpg
[ALARM] Anomali Tespit Edildi! ID: 4 | Sıcaklık: 36.9°C -> /content/anomalies/anomali_ID4_36.9C.jpg
[ALARM] Anomali Tespit Edildi! ID: 22 | Sıcaklık: 36.5°C -> /content/anomalies/anomali_ID22_36.5C.jpg
[ALARM] Anomali Tespit Edildi! ID: 2 | Sıcaklık: 36.6°C -> /content/anomalies/anomali_ID2_36.6C.jpg
[ALARM] Anomali Tespit Edildi! ID: 37 | Sıcaklık: 36.8°C -> /content/anomalies/anomali_ID37_36.8C.jpg
[ALARM] Anomali Tespit Edildi! ID: 3 | Sıcaklık: 36.9°C -> 